# Domain Decider → Research Module

Open this project in its VS Code Dev Container and select **SedAI Docker — Python 3.12** (`/usr/local/bin/python`).

One cell runs both phases. Blank resume creates `runs/<factsheet>/run_001/`; set an explicit full-run root to resume with frozen inputs/settings. Do not run the same job in two environments. Review public-input consent before execution. Advanced linked/upload-only actions remain in the module APIs and CLI.


In [1]:
from pathlib import Path

from ML.deep_research.workflow import create_run, run_all
from ML.deep_research.domain_decider.backend.fs import load_json
from ML.deep_research.domain_decider.backend.settings import DOMAIN_PLUGIN_PATH, RUNS_DIR

FACT_SHEET_PATH = Path("inputs/new_fact_sheet.md")
DOMAIN_PLUGIN = DOMAIN_PLUGIN_PATH
REQUIREMENTS_PATH = Path("inputs/requirement.md")
SOURCE_SUGGESTION_PATH = Path("inputs/source_suggestion.md")
RESEARCH_INSTRUCTION_PATH = Path("inputs/user_research_instruction.md")
RESEARCH_CONFIG_PATH = Path("inputs/research_config.json")

RESUME_RUN_PATH = ""  # Explicit full-run root; blank creates the next numbered run
PUBLIC_INPUT_CONFIRMED = True
REASONING_SUMMARIES = True
RETRY_FAILED = False

# Reasoning: low | medium | high | max; verbosity/search_context: low | medium | high
STAGE_SETTINGS = {
    "metadata": {"reasoning": "high", "verbosity": "medium"},
    "design": {"reasoning": "high", "search_context": "medium", "verbosity": "medium"},
    "distribution": {"reasoning": "high", "verbosity": "medium"},
    "source_discovery": {"reasoning": "high", "search_context": "medium", "verbosity": "low"},
    "research": {"reasoning": "high", "verbosity": "medium"},
    "research_search": {"reasoning": "high", "search_context": "medium", "verbosity": "low"},
    "document": {"reasoning": "high", "verbosity": "medium"},
    "summary": {"reasoning": "medium", "verbosity": "medium"},
}

if RETRY_FAILED and not RESUME_RUN_PATH.strip():
    raise ValueError("RETRY_FAILED requires an explicit resume path.")

FULL_RUN = Path(RESUME_RUN_PATH) if RESUME_RUN_PATH.strip() else create_run(
    FACT_SHEET_PATH, domain_plugin=DOMAIN_PLUGIN, requirements=REQUIREMENTS_PATH,
    source_suggestion=SOURCE_SUGGESTION_PATH, research_instruction=RESEARCH_INSTRUCTION_PATH,
    research_config=RESEARCH_CONFIG_PATH, runs_dir=RUNS_DIR,
    stage_settings={**STAGE_SETTINGS, "reasoning_summaries": REASONING_SUMMARIES},
    public_input_confirmed=PUBLIC_INPUT_CONFIRMED,
)
print(f"Run: {FULL_RUN}\nLog: {FULL_RUN / 'run.log'}")
try:
    await run_all(FULL_RUN, retry_failed=RETRY_FAILED)
finally:
    saved = load_json(FULL_RUN / "run.json")
    print(f"Workflow: {saved['status']}")
    for phase, relative in saved.get("phases", {}).items():
        phase_path = FULL_RUN / relative
        if not (phase_path / "run.json").exists():
            print(f"{phase}: not started")
            continue
        state = load_json(phase_path / "run.json")
        print(f"{phase}: {state['status']} — {phase_path}")
        if phase == "research_module":
            print(f"Discovery: {state.get('discovery_status', 'pending')}")
            uploads = state.get("document_uploads", {})
            print(f"Uploads: {uploads.get('status', 'pending')}; {uploads.get('counts', {})}")
            jobs = state.get("research", {}).get("jobs", {})
            print(f"Research: {sum(j.get('status') == 'complete' for j in jobs.values())}/{len(state['domains'])}")
            for job, entry in jobs.items():
                budget = entry.get("budget", {})
                left = budget.get("remaining")
                balance = 'not recorded' if 'remaining' not in budget else ('unlimited' if left is None else left)
                print(f"{job}: {entry['status']}; {budget.get('used', 'not recorded')} used; {balance} remaining")
            print(f"Sources: {phase_path / 'sources'}\nReports: {phase_path / 'research'}")


Run: /app/runs/new_fact_sheet/run_003
Log: /app/runs/new_fact_sheet/run_003/run.log


/usr/local/lib/python3.12/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `ResponseOutputMessage` - serialized value may not be as expected [field_name='output', input_value=ResponseFunctionWebSearch... type='web_search_call'), input_type=ResponseFunctionWebSearch])
  PydanticSerializationUnexpectedValue(Expected `ResponseFileSearchToolCall` - serialized value may not be as expected [field_name='output', input_value=ResponseFunctionWebSearch... type='web_search_call'), input_type=ResponseFunctionWebSearch])
  PydanticSerializationUnexpectedValue(Expected `ResponseFunctionToolCall` - serialized value may not be as expected [field_name='output', input_value=ResponseFunctionWebSearch... type='web_search_call'), input_type=ResponseFunctionWebSearch])
  PydanticSerializationUnexpectedValue(Expected `ResponseFunctionToolCallOutputItem` - serialized value may not be as expected [field_name='output', input_value=Res

Workflow: complete
domain_decider: complete — /app/runs/new_fact_sheet/run_003/domain_decider
research_module: complete — /app/runs/new_fact_sheet/run_003/research_module
Discovery: complete
Uploads: partial; {'candidate_entries': 55, 'unique_urls': 50, 'uploaded_files': 45, 'failed_urls': 5, 'observations': 0}
Research: 10/10
source_finder/000001: complete; 20 used; 60 remaining
source_finder/000002: complete; 35 used; 45 remaining
source_finder/000003: complete; 24 used; 56 remaining
source_finder/000004: complete; 22 used; 58 remaining
source_finder/000005: complete; 28 used; 52 remaining
source_finder/000006: complete; 28 used; 52 remaining
source_finder/000007: complete; 22 used; 58 remaining
source_finder/000008: complete; 24 used; 56 remaining
source_finder/000009: complete; 33 used; 47 remaining
source_finder/000010: complete; 30 used; 50 remaining
Sources: /app/runs/new_fact_sheet/run_003/research_module/sources
Reports: /app/runs/new_fact_sheet/run_003/research_module/researc